# HPO Dual-View SARF (Average Fusion) (v2) - head search

The encoder side is fixed (`config.FIXED_ENCODER`: batch 32, encoder_lr 1e-5,
weight_decay 0.01, warmup 0.1) - the established stable region for XLM-R-large
(Devlin et al., 2019; Liu et al., 2019; Mosbach et al., 2021). Only the
architecture-specific head is searched: `head_lr`, `dropout` and `epochs`, over
`HPO_HEAD_TRIALS` trials.

Every architecture with a head of its own runs this same search, over the same
space (`config.HEAD_LR_GRID`, `DROPOUT_GRID`, `EPOCH_MIN..EPOCH_MAX`) and with
the same budget - see `config.HPO_SEARCHES`. Equal search budgets, not a shared
configuration, are what make the comparison fair.

The result is written to `hpo/dual_view_v2/best_params.json`. Two variants
inherit it (`config.HPO_INHERITS`): the gated variant, whose gate introduces no
additional hyperparameter, and the single-view ablation, which uses the same head
- inheriting keeps the comparison against them single-variable.

The validation slice uses `HPO_SLICE_SEED` (not the fold-assignment seed), so it
does not coincide with CV fold 0.


In [ ]:
!pip install optuna transformers datasets scikit-learn pandas matplotlib torch

In [ ]:
import optuna
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
import random
import shutil
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

import sys
sys.path.insert(0, "/content/drive/MyDrive/google_colab/kusa/v2_heldout")
from config import *
import utils_split as u

VARIANT  = "dual_view_v2"
SAVE_DIR = hpo_dir(VARIANT)
print("Variant  :", VARIANT)
print("Output   :", SAVE_DIR)

BERT_MODEL_NAME = "xlm-roberta-large"

N_TRIALS = HPO_HEAD_TRIALS
# EPOCHS/PATIENCE as fixed constants removed - epochs is searched,
# Early stopping removed
# EPOCH_MIN, EPOCH_MAX and FIXED_ENCODER come from config
SEED = TRAIN_SEED                           # from config not hardcoded 42 anymore

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
# Data source and slice
dev = pd.read_csv(DEV_POOL, encoding="utf-8")
dev = dev.loc[:, ~dev.columns.str.contains("^Unnamed")]
dev = dev.dropna(subset=["surface", "lemma"]).reset_index(drop=True)
dev = u.add_group_and_stratum(dev)

hpo_train_df, val_df = u.grouped_holdout(dev, HPO_FRAC, HPO_SLICE_SEED)
hpo_train_df = hpo_train_df.reset_index(drop=True)
val_df       = val_df.reset_index(drop=True)

assert u.no_group_overlap(hpo_train_df, val_df)
print(f"Dev-Pool: {len(dev)} | HPO-Train: {len(hpo_train_df)} | HPO-Slice: {len(val_df)}"
      f" ({len(val_df)/len(dev):.1%})")

tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

In [ ]:
class DualViewSurfaceLemmaDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.surface   = df["surface"].tolist()
        self.lemma     = df["lemma"].tolist()
        self.labels    = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.labels)

    def encode(self, text):
        enc = self.tokenizer(
            text, padding="max_length", truncation=True,
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0)
        }

    def __getitem__(self, idx):
        return {
            "surface": self.encode(self.surface[idx]),
            "lemma":   self.encode(self.lemma[idx]),
            "labels":  torch.tensor(self.labels[idx], dtype=torch.long),
        }


def dual_surface_lemma_collator(features):
    batch = {}
    for key in ["surface", "lemma"]:
        batch[key] = {
            "input_ids":      torch.stack([f[key]["input_ids"]      for f in features]),
            "attention_mask": torch.stack([f[key]["attention_mask"] for f in features])
        }
    batch["labels"] = torch.stack([f["labels"] for f in features])
    return batch

In [ ]:
# Architecture unchanged compared to original.
class DualViewCNNBiLSTMAttention(nn.Module):
    def __init__(self, bert_model_name="xlm-roberta-large", lstm_hidden_dim=128,
                 cnn_filters=200, kernel_sizes=(3, 4, 5), num_classes=3, dropout=0.3):
        super(DualViewCNNBiLSTMAttention, self).__init__()

        base_model = AutoModelForSequenceClassification.from_pretrained(
            bert_model_name, return_dict=True, num_labels=num_classes
        )
        self.encoder = base_model.roberta if hasattr(base_model, "roberta") else base_model.bert
        bert_hidden_dim = self.encoder.config.hidden_size

        self.cross_attn_lemma = nn.MultiheadAttention(
            embed_dim=bert_hidden_dim, num_heads=8, batch_first=True
        )


        self.convs = nn.ModuleList([
            nn.Conv2d(in_channels=1, out_channels=cnn_filters,
                      kernel_size=(k, bert_hidden_dim))
            for k in kernel_sizes
        ])

        self.bilstm = nn.LSTM(
            input_size=bert_hidden_dim, hidden_size=lstm_hidden_dim,
            num_layers=1, bidirectional=True, batch_first=True
        )

        fused_dim = (cnn_filters * len(kernel_sizes)) + (lstm_hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(fused_dim, num_classes)

    def forward(self, surface, lemma):
        surface_mask = surface["attention_mask"]
        lemma_mask   = lemma["attention_mask"]

        surface_out = self.dropout(self.encoder(**surface).last_hidden_state)
        lemma_out   = self.dropout(self.encoder(**lemma).last_hidden_state)

        lemma_key_padding_mask = (lemma_mask == 0)
        lemma_attn, _ = self.cross_attn_lemma(
            query=surface_out, key=lemma_out, value=lemma_out,
            key_padding_mask=lemma_key_padding_mask
        )

        cross_attended = (surface_out + lemma_attn) / 2.0

        pad_mask_2d = (surface_mask == 0).unsqueeze(-1)
        cross_attended = cross_attended.masked_fill(pad_mask_2d, 0.0)
        cross_attended = self.dropout(cross_attended)

        pad_mask_4d = (surface_mask == 0).unsqueeze(1).unsqueeze(-1)
        cnn_input   = surface_out.unsqueeze(1).masked_fill(pad_mask_4d, 0.0)

        cnn_features = []
        for conv in self.convs:
            x = F.relu(conv(cnn_input)).squeeze(3)
            x = F.max_pool1d(x, kernel_size=x.size(2)).squeeze(2)
            cnn_features.append(x)
        cnn_out = torch.cat(cnn_features, dim=1)

        lengths      = surface_mask.sum(dim=1).cpu()
        packed_input = nn.utils.rnn.pack_padded_sequence(
            cross_attended, lengths, batch_first=True, enforce_sorted=False
        )
        packed_output, _ = self.bilstm(packed_input)
        lstm_out, _      = nn.utils.rnn.pad_packed_sequence(
            packed_output, batch_first=True, total_length=surface_mask.size(1)
        )

        input_mask_expanded = surface_mask.unsqueeze(-1).expand(lstm_out.size()).float()
        sum_embeddings = torch.sum(lstm_out * input_mask_expanded, dim=1)
        sum_mask       = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        lstm_pooled    = sum_embeddings / sum_mask

        fused  = torch.cat((cnn_out, lstm_pooled), dim=1)
        output = self.dropout(fused)
        logits = self.fc(output)
        return logits

In [ ]:
# New: epochs comes from the search space; the run trains for that exact number of epochs and
#      returns the F1 of the LAST epoch. Taking the maximum over 8 noisy
#      measurements as target systematically biases the search upward.
def run_trial(train_loader, eval_loader, encoder_lr, head_lr, weight_decay,
              warmup_ratio, dropout, epochs, seed=SEED, trial=None):   # [V2] new epochs parameter
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = DualViewCNNBiLSTMAttention(
        bert_model_name=BERT_MODEL_NAME, num_classes=3, dropout=dropout
    )
    model.to(device)

    encoder_params = list(model.encoder.parameters())
    head_params    = [p for n, p in model.named_parameters() if "encoder" not in n]
    optimizer = optim.AdamW([
        {"params": encoder_params, "lr": encoder_lr},
        {"params": head_params,    "lr": head_lr},
    ], weight_decay=weight_decay)

    total_steps  = len(train_loader) * epochs                      # [V2] epochs instead of EPOCHS
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )
    criterion = nn.CrossEntropyLoss()

    eval_f1 = 0.0                                                  # [V2] no best_eval_f1
    try:
        for epoch in range(epochs):                                # [V2]
            model.train()
            epoch_loss = 0.0
            for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
                surface = {k: v.to(device) for k, v in batch["surface"].items()}
                lemma   = {k: v.to(device) for k, v in batch["lemma"].items()}
                labels  = batch["labels"].to(device)

                optimizer.zero_grad()
                logits = model(surface, lemma)
                loss = criterion(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                epoch_loss += loss.item()

            # Collapse guard: prune diverged trials after the first epoch.
            if epoch == 0 and epoch_loss / len(train_loader) > COLLAPSE_LOSS:
                print("  pruned: epoch-1 loss above ln(3)")
                raise optuna.TrialPruned()

            model.eval()
            eval_preds, eval_labels_list = [], []
            with torch.no_grad():
                for batch in eval_loader:
                    surface = {k: v.to(device) for k, v in batch["surface"].items()}
                    lemma   = {k: v.to(device) for k, v in batch["lemma"].items()}
                    labels  = batch["labels"].to(device)

                    logits = model(surface, lemma)
                    preds = torch.argmax(logits, dim=1)
                    eval_preds.extend(preds.cpu().numpy())
                    eval_labels_list.extend(labels.cpu().numpy())

            eval_f1 = f1_score(eval_labels_list, eval_preds, average="macro")
            print(f"Epoch {epoch+1}/{epochs} | Macro-F1: {eval_f1:.4f}")

            if trial is not None:
                trial.report(eval_f1, step=epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            # [V2] Early stopping block completely removed
    finally:
        try:
            del model, optimizer, scheduler
        except NameError:
            pass
        torch.cuda.empty_cache()

    return eval_f1                                                 # [V2] last epoch instead of best epoch


In [ ]:
def objective(trial):
    # Encoder side is fixed (config.FIXED_ENCODER); only the head is searched.
    batch_size   = FIXED_ENCODER["batch_size"]
    encoder_lr   = FIXED_ENCODER["encoder_lr"]
    weight_decay = FIXED_ENCODER["weight_decay"]
    warmup_ratio = FIXED_ENCODER["warmup_ratio"]

    head_lr = trial.suggest_categorical("head_lr", HEAD_LR_GRID)
    dropout = trial.suggest_categorical("dropout", DROPOUT_GRID)
    epochs  = trial.suggest_int("epochs", EPOCH_MIN, EPOCH_MAX)

    print(f"\nTrial {trial.number}: head_lr={head_lr}, dropout={dropout}, "
          f"epochs={epochs}  (fixed: batch={batch_size}, enc_lr={encoder_lr}, "
          f"wd={weight_decay}, warmup={warmup_ratio})")

    train_loader = DataLoader(DualViewSurfaceLemmaDataset(hpo_train_df, tokenizer),
                              batch_size=batch_size, shuffle=True,
                              collate_fn=dual_surface_lemma_collator)
    val_loader   = DataLoader(DualViewSurfaceLemmaDataset(val_df, tokenizer),
                              batch_size=batch_size, shuffle=False,
                              collate_fn=dual_surface_lemma_collator)

    return run_trial(train_loader, val_loader, encoder_lr, head_lr, weight_decay,
                     warmup_ratio, dropout, epochs,
                     seed=SEED + trial.number, trial=trial)


In [ ]:
LOCAL_DB  = "/content/study_local.db"
REMOTE_DB = study_db(VARIANT)               # [V2] v2_heldout/hpo/<variant>/study.db

if os.path.exists(REMOTE_DB):
    shutil.copy(REMOTE_DB, LOCAL_DB)
    print("Existing study retrieved from Drive:", REMOTE_DB)

storage = optuna.storages.RDBStorage(               # [V2]
    f"sqlite:///{LOCAL_DB}",
    engine_kwargs={"connect_args": {"timeout": 100}},
)

study = optuna.create_study(
    study_name=STUDY_NAMES[VARIANT],        # [V2] new name, no collision with old trials
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=HPO_SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    storage=storage,
    load_if_exists=True,
)

def sync_to_drive(study, trial):
    shutil.copy(LOCAL_DB, REMOTE_DB)

FINISHED_STATES = (
    optuna.trial.TrialState.COMPLETE,
    optuna.trial.TrialState.PRUNED,
    optuna.trial.TrialState.FAIL,
)
counted = len([t for t in study.trials if t.state in FINISHED_STATES])
remaining = max(0, N_TRIALS - counted)
print(f"Completed trials: {counted} | remaining: {remaining}")

if remaining > 0:
    study.optimize(objective, n_trials=remaining, callbacks=[sync_to_drive])
else:
    print("All trials already completed.")
shutil.copy(LOCAL_DB, REMOTE_DB)

print("\n========== HPO Results ==========")
print(f"Best Macro-F1 on HPO slice: {study.best_value:.4f}")
for k, v in study.best_params.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# Original: the best parameters were only printed and manually transferred.
# New: saved as JSON so kusa_*_cv_v2 reads them directly - especially
#      the number of epochs needed there without early stopping.
best = dict(study.best_params)              # [V2] head params only
best.update(FIXED_ENCODER)                  # encoder side was fixed, not searched
best["_hpo_slice_seed"] = HPO_SLICE_SEED
best["_variant"]     = VARIANT
best["_study"]       = STUDY_NAMES[VARIANT]
best["_best_value"]  = float(study.best_value)
best["_n_trials"]    = len([t for t in study.trials if t.state in FINISHED_STATES])
best["_hpo_frac"]    = HPO_FRAC
best["_hpo_seed"]    = HPO_SEED

with open(os.path.join(SAVE_DIR, "best_params.json"), "w", encoding="utf-8") as f:
    json.dump(best, f, indent=2)
print(json.dumps(best, indent=2))

try:
    tdf = study.trials_dataframe()
    tdf = tdf[tdf["state"] == "COMPLETE"].reset_index(drop=True)
    if len(tdf) > 0:
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(tdf["number"], tdf["value"], marker="o")
        ax.set_xlabel("Trial")
        ax.set_ylabel("Macro-F1 (HPO-Slice)")
        ax.set_title(f"Optimization History - {VARIANT}")
        ax.grid()
        plt.savefig(os.path.join(SAVE_DIR, "optimization_history.png"), dpi=150,
                    bbox_inches="tight")
        plt.show()
except Exception as e:
    print("Visualization failed:", e)

assert_test_untouched(globals())            # [V2] Protection: Test set untouched